# DNAK against k-d tree filtering (Pelleg–Moore) and dual-tree k-means

**Purpose.** Run the one experiment that the revised manuscript lists as missing (Section 6, "Baselines"): the exact, tree-based k-means methods that index the *data* with a k-d tree, under the same protocol as Table 8 of the paper (same k-means++ centroids, same session, alternating order, identical-output check).

**Baselines.** `mlpack` 4.x ships C++ implementations of the Pelleg–Moore k-d tree ("blacklisting") algorithm (`pelleg-moore`), Curtin's dual-tree k-means (`dualtree`), Elkan (`elkan`), Hamerly (`hamerly`) and plain Lloyd (`naive`). They are called from Python through the official bindings; the timing includes the binding overhead (a copy of the data into Armadillo), which is small compared with the clustering time at these sizes and works against the baselines, not against DNAK.

**What is checked.** For every run the final labels and centroids of each baseline are compared with those of DNAK (which is itself checked against Lloyd's algorithm in the main benchmark). mlpack stops when the centroid movement falls below an internal threshold and counts the initial assignment as an iteration, so its iteration count is normally one higher than DNAK's; a run is "identical" when every label agrees. Runs that are not identical are reported, not hidden: in a test run of this notebook, mlpack's `dualtree` and `hamerly` did not reproduce Lloyd's output on a planar k = 512 problem, while `pelleg-moore`, `elkan` and `naive` did. Only baselines that give identical output can be used in the paper's table.

**Runtime.** Use a standard CPU runtime (Runtime → Change runtime type → CPU). Expect roughly 1–2 hours for the default configuration list; the full Uber problem (last entry, optional) adds about 30–60 minutes. Results are appended to a CSV after every run, so the notebook can be interrupted and restarted.

**Important detail.** mlpack modifies the `initial_centroids` array in place. Every call therefore receives a fresh copy of `C0`; do not remove the `.copy()`.


In [ ]:
# 1. Hardware and software of this session (paste the CPU line and RAM into Section 5.1 of the paper)
!lscpu | grep -E "Model name|^CPU\(s\)|Thread\(s\) per core|Core\(s\) per socket"
!free -g | head -2
!nproc
import platform, sys
print(platform.platform()); print(sys.version)

In [ ]:
# 2. Install mlpack (C++ k-means with Pelleg-Moore, dual-tree, Elkan, Hamerly), wurlitzer (captures mlpack's log), and the DNAK package
!pip -q install mlpack wurlitzer numba scipy pandas scikit-learn
import os, sys
if not os.path.isdir("dnak-kmeans"):
    !git clone -q https://github.com/barunkrpaul/dnak-kmeans.git
os.chdir("dnak-kmeans")
sys.path.insert(0, os.path.abspath("src"))
import numpy, scipy, numba, mlpack
print("numpy", numpy.__version__, "scipy", scipy.__version__, "numba", numba.__version__, "mlpack", mlpack.__version__)

In [ ]:
# 3. Imports, warm-up (compiles the Numba kernels so that compilation time is not measured) and the mlpack wrapper
import time, re, csv, math
import numpy as np, pandas as pd, mlpack
from wurlitzer import pipes
from dnak_kmeans import run_algo, warmup, make_blobs, kmeanspp_init, load_uber, load_road
warmup()

def run_dnak(X, C0, max_iter=500):
    r = run_algo("DNAK-6", X, C0.copy(), max_iter)        # DNAK-6 in the code = DNAK in the paper
    return dict(labels=r["labels"], C=r["C"], time=r["time"], iters=r["iters"], dc=r["comps"])

def run_mlpack(X, C0, algo, max_iter=500):
    Xc = np.ascontiguousarray(X, dtype=np.float64)
    C = np.ascontiguousarray(C0.copy(), dtype=np.float64)   # mlpack overwrites this array: always pass a copy
    with pipes() as (out, err):
        t0 = time.perf_counter()
        o = mlpack.kmeans(input_=Xc, clusters=C.shape[0], initial_centroids=C,
                          algorithm=algo, max_iterations=max_iter,
                          allow_empty_clusters=True,       # same rule as DNAK/Lloyd here: an empty cluster keeps its centroid
                          labels_only=True, verbose=True)
        t = time.perf_counter() - t0
    log = out.read() + err.read()
    m = re.search(r"converged after (\d+) iterations", log)
    dc = re.search(r"(\d+) distance calculations", log)
    return dict(labels=np.asarray(o["output"]).ravel().astype(np.int64),
                C=np.asarray(o["centroid"], dtype=np.float64), time=t,
                iters=int(m.group(1)) if m else -1, dc=int(dc.group(1)) if dc else -1)

def same_output(r, ref):
    nd = int((r["labels"] != ref["labels"]).sum())
    cd = float(np.abs(r["C"] - ref["C"]).max()) if r["C"].shape == ref["C"].shape else float("inf")
    return nd == 0, nd, cd

print("ready")

In [ ]:
# 4. Configurations: the rows of Table 8 of the paper. Data seeds follow experiments/run_dnak6_vs_original.py;
#    the k = 1024 sets use the same generator with the seeds written below (state them in the paper).
CONFIGS = [
    ("synth-2D-k128",  lambda: make_blobs(100000, 2, 128, 23),    128),
    ("sweep-2D-k512",  lambda: make_blobs(100000, 2, 512, 2512),  512),
    ("sweep-2D-k1024", lambda: make_blobs(100000, 2, 1024, 21024), 1024),
    ("sweep-3D-k512",  lambda: make_blobs(100000, 3, 512, 3512),  512),
    ("sweep-3D-k1024", lambda: make_blobs(100000, 3, 1024, 31024), 1024),
    ("uber-k256",      lambda: load_uber(200000, seed=0),         256),
    ("road-k64",       load_road,                                 64),
    # ("uber-full-k1024", lambda: load_uber(None),               1024),   # optional: n = 1,511,443; uncomment for the last row of Table 8
]
SEEDS   = [101, 202, 303]        # k-means++ seeds, as in the paper
REPEATS = 2                      # each (config, seed) is run twice, methods alternated
METHODS = ["pelleg-moore", "dualtree", "elkan", "hamerly"]   # mlpack algorithms; 'naive' (mlpack's Lloyd) is added below for small problems
NAIVE_MAX_NK = 2.0e8             # run mlpack 'naive' only when n*k <= this (it is O(nk) per iteration)
OUT = "results/raw_kdtree_colab.csv"
FIELDS = ["config", "n", "d", "k", "seed", "rep", "method", "iters", "dc", "time_s", "identical", "n_label_diff", "max_centroid_diff"]
os.makedirs("results", exist_ok=True)
done = set()
if os.path.exists(OUT):
    for row in csv.DictReader(open(OUT)):
        done.add((row["config"], int(row["seed"]), int(row["rep"]), row["method"]))
print(len(done), "rows already present")

In [ ]:
# 5. Main loop (resumable). One CSV row per run. Methods are alternated so that machine drift affects all of them alike.
with open(OUT, "a", newline="") as f:
    w = csv.DictWriter(f, fieldnames=FIELDS)
    if f.tell() == 0:
        w.writeheader()
    for name, loader, k in CONFIGS:
        X = loader(); n, d = X.shape
        methods = METHODS + (["naive"] if n * k <= NAIVE_MAX_NK else [])
        for seed in SEEDS:
            C0 = kmeanspp_init(X, k, seed)
            for rep in range(REPEATS):
                order = ["DNAK"] + methods if rep % 2 == 0 else methods[::-1] + ["DNAK"]
                todo = [m for m in order if (name, seed, rep, m) not in done]
                if not todo:
                    continue
                ref = run_dnak(X, C0)          # reference output for this (config, seed); DNAK is itself checked against Lloyd elsewhere
                for m in order:
                    if (name, seed, rep, m) in done:
                        continue
                    r = ref if m == "DNAK" else run_mlpack(X, C0, m)
                    ident, nd, cd = same_output(r, ref)
                    row = dict(config=name, n=n, d=d, k=k, seed=seed, rep=rep, method=m, iters=r["iters"], dc=r["dc"],
                               time_s=f"{r['time']:.4f}", identical=ident, n_label_diff=nd, max_centroid_diff=f"{cd:.3e}")
                    w.writerow(row); f.flush(); done.add((name, seed, rep, m))
                    print(f"{name:16s} seed {seed} rep {rep} {m:13s} {r['time']:8.3f} s  iters {r['iters']:4d}  identical={ident}  (label diffs {nd})", flush=True)
print("finished:", OUT)

In [ ]:
# 6. Summary in the layout of Table 8: mean time, geometric mean of the per-run ratio (baseline / DNAK), runs on which DNAK was faster,
#    and the number of runs on which the baseline reproduced DNAK's output. Only baselines with identical output on every run belong in the paper.
df = pd.read_csv(OUT)
df["time_s"] = df["time_s"].astype(float)
ref = df[df.method == "DNAK"].set_index(["config", "seed", "rep"])["time_s"]
rows = []
for cfg, g in df.groupby("config", sort=False):
    out = {"config": cfg, "DNAK (s)": g[g.method == "DNAK"]["time_s"].mean()}
    for m in [x for x in g.method.unique() if x != "DNAK"]:
        gm = g[g.method == m].set_index(["seed", "rep"])
        ratio = gm["time_s"] / ref.loc[cfg].reindex(gm.index)
        out[f"{m} (s)"] = gm["time_s"].mean()
        out[f"{m}/DNAK"] = math.exp(np.log(ratio).mean())
        out[f"{m} wins"] = f"{int((ratio > 1).sum())}/{len(ratio)}"
        out[f"{m} identical"] = f"{int(gm['identical'].sum())}/{len(gm)}"
    rows.append(out)
summary = pd.DataFrame(rows)
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 40)
print(summary.round(3).to_string(index=False))
summary.to_csv("results/summary_kdtree_colab.csv", index=False)

In [ ]:
# 7. LaTeX rows for a new table (or new columns of Table 8): DNAK, Pelleg-Moore and whichever other baseline was exact on every run.
def latex_rows(summary, baselines=("pelleg-moore", "elkan")):
    lines = []
    for _, r in summary.iterrows():
        cells = [r["config"], f"{r['DNAK (s)']:.2f}"]
        for b in baselines:
            if f"{b} (s)" in r and not pd.isna(r[f"{b} (s)"]):
                cells += [f"{r[f'{b} (s)']:.2f}", f"{r[f'{b}/DNAK']:.2f}", r[f"{b} wins"].replace("/", " of ")]
            else:
                cells += ["--", "--", "--"]
        lines.append(" & ".join(cells) + r" \\")
    return "\n".join(lines)
print(latex_rows(summary))

In [ ]:
# 8. Optional: peak memory of each method on the largest problem, measured in a fresh process with /usr/bin/time -v
#    (within one process the maximum RSS is monotone, so separate processes are needed).
script = '''
import sys, numpy as np, time
sys.path.insert(0, "src")
from dnak_kmeans import run_algo, warmup, make_blobs, kmeanspp_init
import mlpack
method = sys.argv[1]
X = make_blobs(100000, 2, 1024, 21024); C0 = kmeanspp_init(X, 1024, 101)
if method == "DNAK":
    warmup(); r = run_algo("DNAK-6", X, C0.copy()); print("time", r["time"])
else:
    t0 = time.perf_counter()
    mlpack.kmeans(input_=np.ascontiguousarray(X), clusters=1024, initial_centroids=np.ascontiguousarray(C0.copy()),
                  algorithm=method, max_iterations=500, allow_empty_clusters=True, labels_only=True)
    print("time", time.perf_counter() - t0)
'''
open("peak_mem_one.py", "w").write(script)
for m in ["DNAK", "pelleg-moore", "elkan"]:
    print("==", m)
    !/usr/bin/time -v python peak_mem_one.py {m} 2>&1 | grep -E "time |Maximum resident"


## What to put in the paper

1. Section 5.1: the CPU and RAM lines printed by cell 1.
2. Section 5.7 / Table 8: add the Pelleg–Moore column (and any other mlpack baseline that was *identical on every run*; print `summary` to see which). Report, as for Exponion and Ball k-means, the mean time, the geometric-mean ratio and the number of runs on which DNAK was faster. If a baseline was not identical on some run, say so in the text and leave it out of the table.
3. Section 6, paragraph "Baselines": replace the sentence that calls the filtering comparison "the most important missing experiment" by the result, whatever it is. If Pelleg–Moore was faster in some configuration, say so; the paper's claim is about the regime, not about winning everywhere.
4. Data availability: commit `results/raw_kdtree_colab.csv`, `results/summary_kdtree_colab.csv` and this notebook to the repository (`experiments/`), and bump the tag.

The data seeds for the k = 1024 synthetic sets (21024 and 31024 above) are the only new experimental parameters; state them in Section 5.1 or in the caption of Table 8.
